**CI twin of `ch05-loss-functions.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

digits = load_digits()
Xtr, Xte, ytr, yte = train_test_split(
    digits.data, digits.target, test_size=0.25,
    random_state=42, stratify=digits.target)
net = MLPClassifier(hidden_layer_sizes=(16,), activation="relu",
                    random_state=0, max_iter=2000).fit(Xtr, ytr)
W1, W2 = net.coefs_
b1, b2 = net.intercepts_

def forward(X):
    return np.maximum(0, X @ W1 + b1) @ W2 + b2

logits = forward(Xte[[0]])[0]

exps = np.exp(logits)
probs = exps / exps.sum()

for k in range(10):
    print(f"class {k}: logit {logits[k]:6.1f}  ->  p = {probs[k]:.3f}")
print(f"\nsum of probabilities: {probs.sum():.6f}")

In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print(f"naive exp([1000, 999]) = {np.exp(np.array([1000.0, 999.0]))}")

def softmax(z):
    shifted = z - z.max()          # the trick: subtract the max first
    e = np.exp(shifted)
    return e / e.sum()

print(f"stable softmax([1000, 999]) = "
      f"{np.round(softmax(np.array([1000.0, 999.0])), 4)}")
print(f"same answer on the real logits: "
      f"{np.allclose(probs, softmax(logits))}")

In [ ]:
import matplotlib.pyplot as plt

p_axis = np.linspace(0.001, 1, 300)
fig, ax = plt.subplots(figsize=(4.6, 3))
ax.plot(p_axis, -np.log(p_axis))
for pt in (1.0, 0.5, 0.1):
    ax.plot(pt, -np.log(pt), "o", color="crimson")
    ax.annotate(f"p={pt}: {-np.log(pt):.2f}", (pt, -np.log(pt)),
                textcoords="offset points", xytext=(6, 6), fontsize=8)
ax.set_xlabel("probability given to the truth")
ax.set_ylabel("loss  −log(p)")
plt.show()

print(f"our hesitant 1:  -log(0.763) = {-np.log(0.763):.3f}")

In [ ]:
L = forward(Xte)
P = np.exp(L - L.max(axis=1, keepdims=True))
P = P / P.sum(axis=1, keepdims=True)
losses = -np.log(P[np.arange(len(yte)), yte])

print(f"mean loss over 450 digits: {losses.mean():.3f}")
print(f"median digit's loss: {np.median(losses):.4f}  (supremely confident)")

shock = int(np.argmax(losses))
fig, ax = plt.subplots(figsize=(2, 2))
ax.imshow(Xte[shock].reshape(8, 8), cmap="gray_r")
ax.axis("off")
plt.show()
print(f"the shock: true {yte[shock]}, predicted "
      f"{int(np.argmax(L[shock]))}, p(truth) = "
      f"{P[shock, yte[shock]]:.5f}, loss = {losses[shock]:.2f}")

In [ ]:
print("p(truth)   MSE's push        CE's push")
for p_val in (0.5, 0.1, 0.001):
    mse_push = (p_val - 1) * p_val * (1 - p_val)   # carries sigma's slope
    ce_push = p_val - 1                            # error, plain
    print(f"  {p_val:<8} {mse_push:+.6f}        {ce_push:+.3f}")

In [ ]:
shifted = logits - logits.max()
e = np.exp(shifted)
probs = e / e.sum()

run_tests([
    ("the odds sum to one", round(float(probs.sum()), 6), 1.0),
    ("judge 1's shout, as a probability", round(float(probs[1]), 3), 0.763),
    ("the buried judge", round(float(probs[2]), 6), 0.0),
    ("overflow-proof by construction", float(shifted.max()), 0.0),
])

In [ ]:
def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

def cross_entropy(probs, true_class):
    return float(-np.log(probs[true_class]))

run_tests([
    ("no opinion, equal odds", softmax(np.array([0.0, 0.0])).tolist(),
     [0.5, 0.5]),
    ("survives huge logits", round(float(
        softmax(np.array([1000.0, 999.0]))[0]), 4), 0.7311),
    ("ten-way total ignorance costs log(10)", round(
        cross_entropy(np.full(10, 0.1), 3), 4), 2.3026),
    ("right and nearly certain costs almost nothing", round(
        cross_entropy(np.array([0.01, 0.99]), 1), 4), 0.0101),
    ("confidently wrong is ruinous", round(
        cross_entropy(np.array([0.999, 0.001]), 1), 4), 6.9078),
], tol=1e-9)